# Chuwi — comparación de modelos de emoción facial infantil

Comparación reproducible de MobileNetV3-Large, EfficientNet-B0, MobileViT-XXS y DeiT-Tiny usando exclusivamente Children_Emotion.

Métrica principal: Macro-F1. También se registran Accuracy, Weighted-F1, F1 por clase, matriz de confusión, parámetros y tiempos.

In [ ]:
!pip -q install 'timm>=1.0,<2' pandas scikit-learn pillow pyyaml


In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))


In [ ]:
import zipfile, pathlib, shutil
zip_name = next(iter(uploaded))
root = pathlib.Path('/content/chuwi_dataset')
if root.exists(): shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(zip_name) as z: z.extractall(root)
dataset_root = next(root.glob('*'))
manifest = next(dataset_root.rglob('emotion_manifest.csv'))
print('Dataset:', dataset_root)
print('Manifest:', manifest)


In [ ]:
import pandas as pd
df = pd.read_csv(manifest)
print(df.shape)
print(df.groupby(['source','split']).size())
print(df['class'].value_counts())


In [ ]:
!git clone -q https://github.com/Isa18121999/Multimodal-Affective-Robot.git /content/chuwi_repo
%cd /content/chuwi_repo
!python scripts/benchmark_facial_models.py --dataset-root "$dataset_root" --manifest "$manifest" --output /content/chuwi_benchmark --models mobilenet_v3_large efficientnet_b0 mobilevit_xxs deit_tiny --epochs 20 --batch-size 32 --lr 1e-4 --weight-decay 1e-4 --patience 4 --seed 42


In [ ]:
results = pd.read_csv('/content/chuwi_benchmark/benchmark_results.csv')
display(results[['model','parameters','best_validation_macro_f1','test_accuracy','test_macro_f1','test_weighted_f1','training_seconds','test_seconds','test_samples_per_second']].sort_values('test_macro_f1', ascending=False))


## Interpretación

No se selecciona un modelo solo por Accuracy. Se revisa Macro-F1, F1 por emoción, matriz de confusión y costo computacional. Después se valida la inferencia del modelo seleccionado en Raspberry Pi 5.

Este experimento evalúa desempeño técnico; no demuestra eficacia clínica.

In [ ]:
results.to_csv('/content/chuwi_model_comparison.csv', index=False)
from google.colab import files
files.download('/content/chuwi_model_comparison.csv')
